# 📊 دراسة وتحليل الأداء والطلب لشركة توصيل وجبات جاهزة (Food Demand Analysis)

**المؤلف:** خبير تحليل البيانات وذكاء الأعمال  
**المستودع:** `food-demand-analysis`  
**الهدف:** الإجابة الرياضية والإحصائية الشاملة على أسئلة إدارة العمليات والتسويق، واختبار أثر العروض الترويجية والخصومات، وحساب مؤشرات التذبذب (CV) لضبط المخزون الاحتياطي.

In [1]:
# 1. استيراد المكتبات الأساسية وضبط البيئة
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# ضبط التنسيق الجمالي للرسومات البيانية
plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", None)
print("Libraries successfully imported!")


Libraries successfully imported!


## 2. تحميل البيانات وربط الجداول (Data Loading & Merging)
تطبيقاً لقواعد العمل في `AGENTS.md`:
* لا تعديل على البيانات الخام في `data/raw/`.
* التحقق الفوري من تطابق عدد الصفوف ومجموع الطلبات قبل وبعد الربط.

In [2]:
# تحميل الجداول الثلاثة من data/raw
train_path = "../data/raw/train.csv"
centers_path = "../data/raw/fulfilment_center_info.csv"
meals_path = "../data/raw/meal_info.csv"

train = pd.read_csv(train_path)
centers = pd.read_csv(centers_path)
meals = pd.read_csv(meals_path)

initial_rows = len(train)
initial_orders = train["num_orders"].sum()
print(f"Train Rows: {initial_rows:,} | Initial Sum of Orders: {initial_orders:,}")

# ربط الجداول بنظام Left Join
df = train.merge(centers, on="center_id", how="left").merge(meals, on="meal_id", how="left")

# التحقق الصارم من الحوكمة
assert len(df) == initial_rows, "Row count mismatch after merge!"
assert df["num_orders"].sum() == initial_orders, "Sum of orders mismatch after merge!"
print(f"Merged Rows: {len(df):,} | Final Sum of Orders: {df['num_orders'].sum():,}")
print(">> Verified: Perfect Join with 0 row loss and 0 order duplication.")
df.head(3)


Train Rows: 456,548 | Initial Sum of Orders: 119,557,485
Merged Rows: 456,548 | Final Sum of Orders: 119,557,485
>> Verified: Perfect Join with 0 row loss and 0 order duplication.


## 3. هندسة المتغيرات المشتقة والمؤشرات المالية (Feature Engineering & KPIs)
وفقاً لقرارات التنظيف في `docs/cleaning_decisions.md`:
* الإيرادات: `revenue = checkout_price * num_orders`
* نسبة الخصم: `discount_rate = (base_price - checkout_price) / base_price`

In [3]:
# هندسة المتغيرات المشتقة
df["revenue"] = df["checkout_price"] * df["num_orders"]
df["discount_amount"] = np.maximum(0, df["base_price"] - df["checkout_price"])
df["discount_rate"] = np.maximum(0, (df["base_price"] - df["checkout_price"]) / df["base_price"])
df["has_discount"] = (df["base_price"] > df["checkout_price"]).astype(int)
df["has_promo"] = ((df["emailer_for_promotion"] == 1) | (df["homepage_featured"] == 1)).astype(int)

total_orders = df["num_orders"].sum()
total_revenue = df["revenue"].sum()
avg_discount = df["discount_rate"].mean() * 100
promo_rows_pct = df["has_promo"].mean() * 100

print("--- المؤشرات المالية والتشغيلية الكبرى ---")
print(f"إجمالي الطلبات: {total_orders:,} طلب")
print(f"إجمالي الإيرادات: {total_revenue:,.2f} ريال/دولار")
print(f"متوسط نسبة الخصم: {avg_discount:.2f}%")
print(f"نسبة الصفوف ذات العروض التسويقية: {promo_rows_pct:.2f}%")


--- المؤشرات المالية والتشغيلية الكبرى ---
إجمالي الطلبات: 119,557,485 طلب
إجمالي الإيرادات: 31,922,771,040.11 ريال/دولار
متوسط نسبة الخصم: 5.90%
نسبة الصفوف ذات العروض التسويقية: 14.82%


## 4. إجابة أسئلة التحليل الستة (The 6 Analysis Questions)
### السؤال 1: كيف يتغير إجمالي الطلب من أسبوع لآخر؟

In [4]:
weekly = df.groupby("week").agg(
    total_orders=("num_orders", "sum"),
    total_revenue=("revenue", "sum")
).reset_index()

mean_w = weekly["total_orders"].mean()
std_w = weekly["total_orders"].std()
min_w = weekly.loc[weekly["total_orders"].idxmin()]
max_w = weekly.loc[weekly["total_orders"].idxmax()]

print(f"متوسط الطلب الأسبوعي: {mean_w:,.1f} +/- {std_w:,.1f}")
print(f"أدنى أسبوع طلباً: الأسبوع {int(min_w['week'])} ({int(min_w['total_orders']):,} طلب)")
print(f"أعلى أسبوع طلباً: الأسبوع {int(max_w['week'])} ({int(max_w['total_orders']):,} طلب)")

# رسم خط الطلب الأسبوعي
plt.figure(figsize=(12, 4))
plt.plot(weekly["week"], weekly["total_orders"] / 1e3, color="#1f77b4", lw=2, label="الطلب الأسبوعي (بالآلاف)")
plt.axhline(mean_w / 1e3, color="gray", linestyle=":", label=f"المتوسط ({mean_w/1e3:.1f}K)")
plt.title("تذبذب الطلب الأسبوعي حول متوسط 824.5 ألف طلب مع قمة في الأسبوع 48", fontsize=12, fontweight="bold")
plt.xlabel("رقم الأسبوع (1 إلى 145)")
plt.ylabel("الطلبات (بالآلاف)")
plt.legend()
plt.tight_layout()
plt.show()


متوسط الطلب الأسبوعي: 824,534.4 +/- 125,670.3
أدنى أسبوع طلباً: الأسبوع 62 (380,065 طلب)
أعلى أسبوع طلباً: الأسبوع 48 (1,303,457 طلب)


### السؤال 2: أي الفئات والوجبات تصنع معظم الطلب؟ (تحليل باريتو 80/20)

In [5]:
cat_df = df.groupby("category")["num_orders"].sum().sort_values(ascending=False).reset_index()
cat_df["order_pct"] = (cat_df["num_orders"] / total_orders) * 100
cat_df["cum_pct"] = cat_df["order_pct"].cumsum()

print("--- الفئات الأربع الأكثر طلباً تصنع 75.2% من المبيعات ---")
print(cat_df.head(4))

meal_df = df.groupby(["meal_id", "category", "cuisine"])["num_orders"].sum().sort_values(ascending=False).reset_index()
meal_df["order_pct"] = (meal_df["num_orders"] / total_orders) * 100
meal_df["cum_pct"] = meal_df["order_pct"].cumsum()
meals_80 = meal_df[meal_df["cum_pct"] <= 80]
print(f"عدد الوجبات التي تصنع 80% من الطلب: {len(meals_80)} وجبة من أصل 51 ({len(meals_80)/51*100:.1f}%)")


--- الفئات الأربع الأكثر طلباً تصنع 75.2% من المبيعات ---
    category  num_orders  order_pct    cum_pct
0  Beverages    40480525  33.858629  33.858629
1  Rice Bowl    20874063  17.459436  51.318065
2   Sandwich    17636782  14.751717  66.069782
3      Salad    10944336   9.154037  75.223819
عدد الوجبات التي تصنع 80% من الطلب: 21 وجبة من أصل 51 (41.2%)


### السؤال 3: أي المراكز أعلى أداءً؟ وهل يختلف الأداء حسب نوع المركز؟

In [6]:
type_summary = df.groupby("center_type").agg(
    num_centers=("center_id", "nunique"),
    total_orders=("num_orders", "sum"),
    avg_op_area=("op_area", "mean")
).reset_index()
type_summary["share_pct"] = (type_summary["total_orders"] / total_orders) * 100
print("--- أداء المراكز حسب نوع المركز ---")
print(type_summary)


--- أداء المراكز حسب نوع المركز ---
  center_type  num_centers  total_orders  avg_op_area  share_pct
0      TYPE_A           43      68978517     4.115983  57.694854
1      TYPE_B           15      29996073     4.852873  25.089247
2      TYPE_C           19      20582895     3.271435  17.215898


### السؤال 4: هل ترفع عروض البريد والظهور في الصفحة الرئيسية الطلب؟ وبكم؟ (اختبار Mann-Whitney U)

In [7]:
# 1. اختبار البريد الترويجي (Emailer)
em_1 = df[df["emailer_for_promotion"] == 1]["num_orders"]
em_0 = df[df["emailer_for_promotion"] == 0]["num_orders"]
u_em, p_em = stats.mannwhitneyu(em_1, em_0, alternative="greater")
em_lift_mean = ((em_1.mean() - em_0.mean()) / em_0.mean()) * 100
em_lift_med = ((em_1.median() - em_0.median()) / em_0.median()) * 100

# 2. اختبار الظهور في الصفحة الرئيسية (Homepage Featured)
hp_1 = df[df["homepage_featured"] == 1]["num_orders"]
hp_0 = df[df["homepage_featured"] == 0]["num_orders"]
u_hp, p_hp = stats.mannwhitneyu(hp_1, hp_0, alternative="greater")
hp_lift_mean = ((hp_1.mean() - hp_0.mean()) / hp_0.mean()) * 100
hp_lift_med = ((hp_1.median() - hp_0.median()) / hp_0.median()) * 100

# 3. اختبار القناتين معاً
both_1 = df[(df["emailer_for_promotion"] == 1) & (df["homepage_featured"] == 1)]["num_orders"]
neither = df[(df["emailer_for_promotion"] == 0) & (df["homepage_featured"] == 0)]["num_orders"]
u_both, p_both = stats.mannwhitneyu(both_1, neither, alternative="greater")
both_lift_mean = ((both_1.mean() - neither.mean()) / neither.mean()) * 100
both_lift_med = ((both_1.median() - neither.median()) / neither.median()) * 100

print(f"Emailer Promo: Mean Lift = +{em_lift_mean:.1f}%, Median Lift = +{em_lift_med:.1f}% (p-val = {p_em})")
print(f"Homepage Featured: Mean Lift = +{hp_lift_mean:.1f}%, Median Lift = +{hp_lift_med:.1f}% (p-val = {p_hp})")
print(f"Both Combined: Mean Lift = +{both_lift_mean:.1f}%, Median Lift = +{both_lift_med:.1f}% (p-val = {p_both})")


Emailer Promo: Mean Lift = +175.3%, Median Lift = +174.8% (p-val = 0.0)
Homepage Featured: Mean Lift = +169.1%, Median Lift = +184.6% (p-val = 0.0)
Both Combined: Mean Lift = +286.1%, Median Lift = +288.5% (p-val = 0.0)


### السؤال 5: ما علاقة الخصم بالطلب؟ (ارتباط سبيرمان)

In [8]:
rs, p_rs = stats.spearmanr(df["discount_rate"], df["num_orders"])
print(f"معامل ارتباط سبيرمان (Spearman Correlation): {rs:.4f} (p-value: {p_rs})")

# مقارنة الخصم عند وجود عروض
disc_no_promo = df[df["has_promo"] == 0]["discount_rate"].mean() * 100
disc_with_promo = df[df["has_promo"] == 1]["discount_rate"].mean() * 100
print(f"متوسط نسبة الخصم بدون عروض: {disc_no_promo:.2f}%")
print(f"متوسط نسبة الخصم مع وجود عروض: {disc_with_promo:.2f}%")
print(">> ملاحظة تداخل العوامل: العروض تقترن بخصومات سعرية أعلى بـ 5 أضعاف.")


معامل ارتباط سبيرمان (Spearman Correlation): 0.1579 (p-value: 0.0)
متوسط نسبة الخصم بدون عروض: 3.67%
متوسط نسبة الخصم مع وجود عروض: 18.67%
>> ملاحظة تداخل العوامل: العروض تقترن بخصومات سعرية أعلى بـ 5 أضعاف.


### السؤال 6: أي الوجبات طلبها متذبذب وتحتاج مخزوناً احتياطياً أكبر؟ (معامل التذبذب CV)

In [9]:
meal_vol = df.groupby(["meal_id", "category", "cuisine"]).agg(
    total_orders=("num_orders", "sum"),
    mean_orders=("num_orders", "mean"),
    std_orders=("num_orders", "std")
).reset_index()
meal_vol["cv"] = meal_vol["std_orders"] / meal_vol["mean_orders"]
meal_vol = meal_vol.sort_values(by="cv", ascending=False)

print("--- أعلى 5 وجبات تذبذباً (تحتاج أعلى مخزون أمان وتحكم في التلف) ---")
print(meal_vol.head(5)[["meal_id", "category", "cuisine", "total_orders", "cv"]])

print("\n--- أعلى 5 وجبات استقراراً (أقل تذبذب وأسهل في التنبؤ) ---")
print(meal_vol.tail(5)[["meal_id", "category", "cuisine", "total_orders", "cv"]])


--- أعلى 5 وجبات تذبذباً (تحتاج أعلى مخزون أمان وتحكم في التلف) ---
    meal_id  category      cuisine  total_orders        cv
25     1971  Sandwich      Italian       5728620  1.445328
34     2444   Seafood  Continental        942568  1.423605
10     1445   Seafood  Continental       1059221  1.338016
49     2867   Seafood  Continental        713925  1.268094
35     2490     Salad      Italian       3709125  1.265016

--- أعلى 5 وجبات استقراراً (أقل تذبذب وأسهل في التنبؤ) ---
    meal_id   category  cuisine  total_orders        cv
17     1770    Biryani   Indian        190709  0.711198
6      1247    Biryani   Indian        231897  0.700149
39     2569      Salad  Italian       3334399  0.691547
44     2664      Salad  Italian       3900812  0.625982
0      1062  Beverages  Italian       4712795  0.581228


## 5. نموذج التعلم الآلي للتنبؤ بالطلب (Bonus ML Task)
* التنبؤ بطلبات آخر 10 أسابيع (الأسابيع 136 إلى 145).
* مقارنة Random Forest بخط الأساس (Historical Mean Baseline) بمقياس MAE.

In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

df_ml = pd.get_dummies(df, columns=["center_type", "category", "cuisine"], drop_first=True)
features = ["checkout_price", "base_price", "discount_rate", "emailer_for_promotion", "homepage_featured", "op_area"]
dummy_cols = [c for c in df_ml.columns if any(c.startswith(p) for p in ["center_type_", "category_", "cuisine_"])]
ml_features = features + dummy_cols

train_data = df_ml[df_ml["week"] <= 135]
val_data = df_ml[df_ml["week"] >= 136]

X_train = train_data[ml_features]
y_train = np.log1p(train_data["num_orders"])
X_val = val_data[ml_features]
y_val = val_data["num_orders"]

mean_lookup = train_data.groupby(["center_id", "meal_id"])["num_orders"].mean().to_dict()
val_pred_baseline = val_data.apply(lambda r: mean_lookup.get((r["center_id"], r["meal_id"]), train_data["num_orders"].mean()), axis=1)
mae_baseline = mean_absolute_error(y_val, val_pred_baseline)

rf_model = RandomForestRegressor(n_estimators=50, max_depth=16, min_samples_leaf=10, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_pred = np.maximum(0, np.expm1(rf_model.predict(X_val)))
mae_rf = mean_absolute_error(y_val, rf_pred)

print(f"MAE Baseline (المتوسط التاريخي): {mae_baseline:.2f}")
print(f"MAE Random Forest: {mae_rf:.2f}")
print(f"نسبة التحسن في الدقة: {((mae_baseline - mae_rf)/mae_baseline)*100:.2f}%")


MAE Baseline (المتوسط التاريخي): 94.32
MAE Random Forest: 87.99
نسبة التحسن في الدقة: 6.71%
